# ALD literature trend analysis

OpenAlex에서 ALD precursor 관련 논문 메타데이터를 수집하고, 2015–2019년과 2020년–현재의 제목 키워드 빈도 변화를 비교합니다.

- 네트워크 수집은 `scripts.fetch_openalex`의 timeout, retry, cursor pagination을 사용합니다.
- 기본 결과는 `artifacts/openalex/`에 저장되며 Git에는 포함되지 않습니다.
- 재현에 필요한 검색식과 기간은 CSV 옆 manifest에 기록됩니다.
- `OPENALEX_API_KEY`는 선택 사항이며 환경 변수로만 전달합니다.


In [ ]:
# 1. OpenAlex 문헌 수집
from datetime import datetime, timezone
from pathlib import Path
import os

import pandas as pd

from scripts.fetch_openalex import CSV_COLUMNS, fetch_works, write_outputs

QUERY = "Atomic Layer Deposition Precursor"
START_YEAR = 2015
END_YEAR = datetime.now(timezone.utc).year
MAX_RESULTS = 2_000
OUTPUT_PATH = Path("artifacts/openalex/ald_openalex_data.csv")

records, fetch_metadata = fetch_works(
    query=QUERY,
    start_year=START_YEAR,
    end_year=END_YEAR,
    max_results=MAX_RESULTS,
    min_abstract_chars=50,
    api_key=os.getenv("OPENALEX_API_KEY"),
)
csv_path, manifest_path = write_outputs(records, fetch_metadata, OUTPUT_PATH)
df_all = pd.DataFrame.from_records(records, columns=CSV_COLUMNS)

print(f"수집 기간: {START_YEAR}–{END_YEAR}")
print(f"저장 논문: {len(df_all):,}건")
print(f"CSV: {csv_path}")
print(f"Manifest: {manifest_path}")
display(df_all[["publication_year", "title", "openalex_id"]].head())


In [ ]:
# 2. 데이터 품질 확인
if df_all.empty:
    raise RuntimeError("검색 결과가 없습니다. 검색식과 기간을 확인하세요.")

duplicate_ids = int(df_all["openalex_id"].duplicated().sum())
missing_titles = int(df_all["title"].isna().sum() + df_all["title"].eq("").sum())

print(f"OpenAlex ID 중복: {duplicate_ids}")
print(f"제목 누락: {missing_titles}")
print("연도별 논문 수:")
display(df_all["publication_year"].value_counts().sort_index().to_frame("works"))


In [ ]:
# 3. 시기별 제목 키워드 변화 분석
import re

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer


def clean_title(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[^\w\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


stopwords_final = sorted({
    "a", "an", "and", "area", "as", "at", "atomic", "authors", "based",
    "by", "characteristic", "characteristics", "characterization", "codoped",
    "crystalline", "crystallinity", "doped", "doping", "deposit", "deposited",
    "deposition", "device", "devices", "direct", "effect", "effects", "efficient",
    "enhanced", "facile", "film", "films", "for", "framework", "from", "gas",
    "grown", "growing", "growth", "high", "impact", "in", "influence", "its",
    "large", "layer", "layered", "low", "mechanism", "metal", "new", "nitride",
    "novel", "observed", "of", "on", "our", "oxide", "phase", "plasma",
    "precursor", "precursors", "preparation", "prepared", "process", "processes",
    "promote", "promoted", "properties", "property", "reaction", "reactions",
    "reduction", "reported", "review", "role", "showed", "small", "structure",
    "structures", "studies", "study", "sulfide", "synthesis", "synthesized",
    "temperature", "the", "their", "thermal", "thin", "to", "use", "used",
    "using", "via", "we", "with",
})

df_all["clean_title"] = df_all["title"].apply(clean_title)
period_past = (2015, 2019)
period_recent = (2020, END_YEAR)
df_past = df_all[df_all["publication_year"].between(*period_past)]
df_recent = df_all[df_all["publication_year"].between(*period_recent)]

if df_past.empty or df_recent.empty:
    raise RuntimeError("두 비교 기간 모두에 논문이 있어야 합니다.")

vectorizer = CountVectorizer(
    stop_words=stopwords_final,
    ngram_range=(1, 2),
    min_df=3,
    token_pattern=r"(?u)\b\w\w+\b",
)
vectorizer.fit(df_all["clean_title"])
vocabulary = vectorizer.get_feature_names_out()


def normalized_frequency(frame):
    matrix = vectorizer.transform(frame["clean_title"])
    return np.asarray(matrix.sum(axis=0)).ravel() / len(frame)


frequency_past = normalized_frequency(df_past)
frequency_recent = normalized_frequency(df_recent)
df_trend = pd.DataFrame({
    "Keyword": vocabulary,
    "Change": frequency_recent - frequency_past,
    "Recent_Freq": frequency_recent,
    "Past_Freq": frequency_past,
})

top_rising = df_trend.nlargest(15, "Change")
top_falling = df_trend.nsmallest(5, "Change")
df_viz = pd.concat([top_rising, top_falling], ignore_index=True)
palette = {
    keyword: ("#D32F2F" if change > 0 else "#1976D2")
    for keyword, change in zip(df_viz["Keyword"], df_viz["Change"])
}

plt.figure(figsize=(12, 12))
sns.barplot(
    data=df_viz,
    x="Change",
    y="Keyword",
    hue="Keyword",
    palette=palette,
    legend=False,
)
plt.title(
    f"ALD Research Trends\n"
    f"({period_past[0]}–{period_past[1]}) vs "
    f"({period_recent[0]}–{period_recent[1]})",
    fontsize=16,
)
plt.xlabel("Change in frequency per paper", fontsize=12)
plt.axvline(0, color="black", linewidth=0.8)
plt.grid(axis="x", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

print("급상승 키워드 Top 10")
display(top_rising[["Keyword", "Change"]].head(10))
